# Word sentiment EEG — compact onsets and class ERPs

Use a **CPU** runtime and choose **Runtime → Run all**. Allow Drive mounting. No GitHub token or label editing is needed.

**Compact storage:** saves every word onset, label, exclusion, and all 105-channel positive/negative/neutral class ERP averages. It calculates the same −200 to +800 ms epochs, but does not save the large individual trial EEG arrays. Those trials can be regenerated from the source EEG and saved onsets.

The run checks existing analyses from `v3_word_erp`, copies only verified onset tables and averages, and extracts missing readers. Existing source data and old caches are preserved. Missing source files download in Colab only, one at a time, and are removed from Colab after use.

Outputs are isolated in `Thesis/Results/word_sentiment_eeg/v4_word_erp_compact/`, including a self-contained `compact_results.zip`. Estimated additional storage for all 36 readers and the bundle is about **100–200 MB**, depending on compression. The initial 1 GiB free-space requirement is a safety margin. Pending uploads from previous runs may still consume space; the notebook stops if the mounted Drive reports insufficient room.

The same first-fixation matching, filtering, baseline, rejection, and uncertainty exclusions apply. SR, NR and TSR remain separate. Agent annotations are not human consensus labels. Plots are descriptive fixation-related potentials, without overlap deconvolution or significance claims.

Live logs appear in the second code cell. A heartbeat shows liveness, not percentage completion. Re-running audits the compact checkpoints and keeps each reader/task once.


In [ ]:
# Run all. This cell runs only in Colab; nothing is installed on your Mac.
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, shutil
from IPython.display import Markdown, display

drive.mount('/content/drive')
free_gib = shutil.disk_usage('/content/drive').free / 1024**3
print(f'Drive available: {free_gib:.2f} GiB', flush=True)
if free_gib < 1:
    raise RuntimeError('Mounted Drive still has less than 1 GiB free. Confirm the selected unwanted cache was permanently deleted from Trash, allow storage/sync to update, then rerun. No EEG extraction has started.')
THESIS = Path('/content/drive/MyDrive/Thesis')
REPO = Path('/content/word-sentiment-eeg')
REPO_URL = 'https://github.com/parmisbathayan/word-sentiment-eeg.git'
RUN_TAG = 'v4_word_erp_compact'

# Public repository: clone/update without tokens or credential prompts.
def git(*args):
    result = subprocess.run(
        ['git', '-c', 'credential.helper=', '-c', 'http.extraheader=', *map(str, args)],
        env=dict(os.environ, GIT_TERMINAL_PROMPT='0'), capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('Could not fetch the public GitHub repository. Check the Colab internet connection and any local checkout edits, then rerun. No GitHub token is required.')
if not REPO.exists():
    git('clone', '--depth', '1', '--branch', 'main', REPO_URL, REPO)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('The Colab code checkout has edits. Save them before updating the checkout.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')], check=True)
print('Code version:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Public GitHub code loaded without a token. CPU is sufficient.')
print('COMPACT MODE: keep word onsets and class ERPs; no new individual trial EEG archives.')


In [ ]:
# Extract EEG with live logs and a status message every 30 seconds.
sys.path.insert(0, str(REPO / 'scripts'))
import importlib, colab_progress
run_with_progress = importlib.reload(colab_progress).run_with_progress
command = [sys.executable, '-u', str(REPO / 'run_compact_erp.py'), '--thesis-root', str(THESIS), '--run-tag', RUN_TAG, '--allow-cloud-download']
run_with_progress(command, THESIS / 'Results/word_sentiment_eeg' / RUN_TAG / 'progress.json')


In [ ]:
RUN = THESIS / 'Results/word_sentiment_eeg' / RUN_TAG
if (RUN / 'REPORT.md').exists():
    display(Markdown((RUN / 'REPORT.md').read_text()))
else:
    print('The run is incomplete. See', RUN / 'run.log')
from IPython.display import Image
for task in ['SR', 'NR', 'TSR']:
    figure = RUN / f'plots/{task}_class_erp.png'
    if figure.exists():
        display(Image(filename=str(figure)))
print('Results:', RUN)
print('Portable results:', RUN / 'compact_results.zip')
print('Individual trial EEG arrays are not archived in compact mode; onsets and source hashes support regeneration.')
print('When finished, tell Codex: the word ERP run finished. Codex can inspect the saved report and result tables through Drive.')
